# Semantic Kernel Intent Classifier Evaluation

This notebook evaluates the performance of our intent classification system using Semantic Kernel. We'll analyze accuracy, generate visualizations, and identify areas for improvement.

In [1]:
import json
from typing import Dict, List
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.functions.kernel_function_decorator import kernel_function
from semantic_kernel.functions.kernel_arguments import KernelArguments

from streaming_ordering_chatbot.api.flows.classification_flow_SK import OrderIntentFlowSK
from streaming_ordering_chatbot.api.models import Message

ENDPOINT = "https://t-toluale-0132-resource.openai.azure.com/"
API_KEY = "1BHVs20ZexZ2kRWF4dP7UrK5vt2pTTpcnaXv1qJ5j4hybIdV2fXuJQQJ99BFACHYHv6XJ3w3AAAAACOGhQjE"
DEPLOYMENT_NAME = "gpt-4o"

# Set paths
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
TEST_DATA_PATH = os.path.join(REPO_ROOT, "tests", "data", "intent_test_cases.json")

ModuleNotFoundError: No module named 'streaming_ordering_chatbot'

In [ ]:
class IntentEvaluationPlugin:
    """Semantic Kernel plugin for evaluating intent classification."""
    
    @kernel_function(name="evaluate_accuracy", description="Evaluates the accuracy of intent classification")
    def evaluate_accuracy(self, expected: str, predicted: str) -> float:
        return 1.0 if expected == predicted else 0.0
    
    @kernel_function(name="log_error", description="Logs classification errors with details")
    def log_error(self, message: str, expected: str, predicted: str, scenario: str) -> str:
        return f"Error in {scenario}: Expected {expected}, got {predicted} for message: '{message}'"

class SKIntentEvaluator:
    def __init__(self, endpoint: str, api_key: str, deployment_name: str):
        self.endpoint = endpoint
        self.api_key = api_key
        self.deployment_name = deployment_name
        
        # Initialize Semantic Kernel
        self.kernel = Kernel()
        chat_service = AzureChatCompletion(
            deployment_name=self.deployment_name,
            endpoint=self.endpoint,
            api_key=self.api_key
        )
        self.kernel.add_service(chat_service)
        
        # Add evaluation plugin
        self.kernel.add_plugin(IntentEvaluationPlugin(), "evaluation")
        
        # Initialize classifier
        self.classifier = OrderIntentFlowSK(endpoint, api_key, deployment_name)
        
        # Load test cases
        self.test_cases = self._load_test_cases()
    
    def _load_test_cases(self) -> List[Dict]:
        with open(TEST_DATA_PATH, "r") as f:
            return json.load(f)["test_cases"]
    
    async def evaluate_single_case(self, test_case: Dict) -> Dict:
        """Evaluate a single test case using Semantic Kernel functions."""
        chat_history = [Message(role="user", content=test_case["message"])]
        predicted_intent = await self.classifier(chat_history, test_case["current_order"])
        
        # Use SK function to evaluate accuracy
        accuracy = await self.kernel.invoke(
            plugin_name="evaluation",
            function_name="evaluate_accuracy",
            arguments=KernelArguments(
                expected=test_case["expected_intent"],
                predicted=predicted_intent
            )
        )
        
        result = {
            "message": test_case["message"],
            "scenario": test_case["scenario"],
            "expected": test_case["expected_intent"],
            "predicted": predicted_intent,
            "correct": bool(accuracy)
        }
        
        # Log error if prediction was wrong
        if not accuracy:
            error_log = await self.kernel.invoke(
                plugin_name="evaluation",
                function_name="log_error",
                arguments=KernelArguments(
                    message=test_case["message"],
                    expected=test_case["expected_intent"],
                    predicted=predicted_intent,
                    scenario=test_case["scenario"]
                )
            )
            print(error_log)
        
        return result